# 📊 Applying Machine Learning to a Multi-Factor Equity Strategy

**Author:** _Dr. Jonas Zink, CFA_
**Date:** _September 2025_
**Repository / Colab:** _link here_

## Abstract
This notebook presents the design, training, and evaluation of a machine-learning (ML) model for **dynamic factor allocation** within an equity investment framework. The study is motivated by the limitations of static factor blends (e.g., fixed 50/50 Momentum–Profitability weights), which may fail to adapt to changing market conditions and macroeconomic regimes.

Our approach combines **feature engineering** from both cross-sectional factor information (e.g., momentum, volatility, ROE) and macro/market indicators (e.g., risk-free rate, VIX). For each training month, the ex-post optimal blend between two factors is identified through a grid search that maximizes portfolio returns in the following month. These optimal weights serve as targets for supervised learning models such as Random Forests, LightGBM, XGBoost, and CatBoost.

Once trained, the models generate **time-varying factor weights** out-of-sample, which are then applied to construct portfolios. The resulting ML-driven strategies are benchmarked against three baselines:
1. a **50/50 static blend** of the factors,
2. the **average ML-predicted weight** across the full sample, and
3. the **index portfolio**.

Performance is assessed with annualized return, volatility, Sharpe ratio, and maximum drawdown. Beyond numerical results, the notebook emphasizes transparency of the ML pipeline, visualization of feature importance, and interpretability of the predicted factor weights.

The ultimate goal is to demonstrate whether a machine-learning approach can systematically identify and exploit time-varying relationships between factors and market regimes, thereby delivering superior risk-adjusted performance compared to static allocation rules.


## Objectives
- Build a transparent end-to-end pipeline for ML-based factor weight prediction.
- Benchmark ML-driven factor weights against static heuristics (50/50 and average-ML).
- Evaluate performance using annualized return, volatility, Sharpe ratio, and maximum drawdown.
- Provide modular and reproducible code for future extensions (additional factors, alternative ML models).

## Quick Start
- **Python:** 3.x
- **Key libraries:** `pandas`, `numpy`, `scikit-learn`, `lightgbm`, `xgboost`, `catboost`, `matplotlib`, `tqdm`
- **Data:** provided Excel (`data.xlsx`).
- **Execution:** run cells sequentially from top to bottom; main parameters are defined in **Setup & Parameters**.

> _Tip:_ In Colab, install missing packages in the first cell (`pip install ...`). Paths are set relative by default.

---

## 🗂️ Table of Contents
1. 🔧 Setup & Parameters
2. 💾 Data
3. 🏅 Calculate Stock Ranks (for each factor)
4. 🧠 Model Training
5. 🚀 Model Application
6. 📉 Backtest Calculation
7. 📊 Results
> 7.1 📉 Choice of Training Period
> 7.2 📉 Choice of Feature Variables
> 7.3 📉 Choice of Machine Learning Model
> 7.4 📉 Choice of Relevant Factors
> 7.5 📉 Summary: Best Model

---

## Research Question & Approach
**Question:** Can a supervised ML model learn time-varying blend weights between factors (e.g., Momentum vs. ROE) that outperform static heuristics?

**Approach:** For each training month, we determine the **ex-ante optimal two-factor blend** via grid search. Using contemporaneous features (factor returns, correlations, and regime indicators such as VIX and the risk-free rate), we train a regressor to predict blend weights. Out-of-sample, these predictions are applied to construct portfolios, which are then compared to 50/50, average-ML, and index benchmarks.


## 1. 🔧 Setup & Parameters

In this section we define the **global configuration** of the experiment.
The setup serves two main purposes:

1. **Reproducibility** – all relevant paths, dates, and model parameters are stored in a single dictionary (`params_`).
2. **Clear separation of training and testing** – by explicitly defining start and end periods, we ensure that model evaluation is based on strictly out-of-sample data.

### Key components:
- **Data handling**
  - `update_factor_scores`: whether to recompute factor ranks (momentum, volatility, ROE) or reuse cached values.
  - `price_frequency_str` / `price_frequency_num`: frequency of return calculation (daily in this case, with 252 trading days per year).

- **Time horizons**
  - `training_start_period` / `training_end_period`: defines the in-sample period used to fit the ML model.
  - `test_start_period` / `test_end_period`: defines the strictly out-of-sample backtest window.

- **Model specification**
  - `ml_training_factors`: list of predictor variables (e.g. lagged factor returns, index volatility, correlations, macro proxies).
  - `relevant_factors`: the set of factors to be blended (here: Momentum and ROE).
  - `ml_model`: the learning algorithm (e.g., Random Forest, LightGBM, XGBoost, CatBoost).

Together, these parameters provide a transparent and flexible way to configure different experimental runs. The settings in this notebook use **daily data** with a **Random Forest Regressor**, training from **2007–2012** and testing from **2013–2025**.


In [ ]:
import pandas as pd
from ML_Multifactor import run_ml_backtests

In [ ]:
params_ = {'update_factor_scores': False,
           'price_frequency_str': 'D',
           'price_frequency_num': 252,
           'training_start_period': pd.Timestamp('2007-01-31'),
           'training_end_period': pd.Timestamp('2012-12-31'),
           'test_start_period': pd.Timestamp('2013-01-01'),
           'test_end_period': pd.Timestamp('2025-03-20'),
           'ml_training_factors': ['mom_roe_corr', 'past_mom_return', 'past_roe_return',
                                   'past_index_return', 'past_index_vola', 'rf', 'vix'],
           'relevant_factors': ['mom', 'roe'],
           'ml_model': 'RandomForestRegressor'}

## 2. 💾 Data

This section describes the **input data** used for training and backtesting the multi-factor strategy.
All datasets can either be loaded directly from the **raw Excel file** (`data.xlsx`) for a full refresh.

### Data sources
- **Stock prices**
  Daily adjusted closing prices for all index constituents. Used to calculate returns and momentum metrics.

- **Index weights**
  Benchmark index weights per constituent. These are re-scaled to sum to one and serve as the baseline allocation.

- **Fundamentals (ROE)**
  Company-level return on equity values. Forward- and backward-filled to ensure data availability across time.

- **Macro/market variables**
  - `RF`: risk-free rate (in percent, converted to decimal form).
  - `VIX`: implied volatility index (as a measure of market uncertainty).

### Processing steps
1. **Data loading**
   - Load raw Excel sheets, clean, align dates.

2. **Resampling**
   Prices are resampled to the frequency defined in `params_` (daily in this case).

3. **Alignment & normalization**
   - Index weights are forward-filled to match all trading days.
   - All weights are normalized so that they sum to 1 per date.

4. **Return calculation**
   Stock-level returns are computed as simple percentage changes.

---

📌 **Note:** Having a consistent and aligned dataset across **prices, index weights, fundamentals, and macro indicators** is crucial to avoid look-ahead bias and to ensure valid backtesting results.


## 3. 🏅 Calculate Stock Ranks (for each factor)

In this step we compute **factor-based stock scores** and assign each stock to quintiles.
These ranks form the foundation of the later ML-based factor integration.

### Factors considered
- **Momentum (MOM)** – 12-month price performance

  $$\text{Momentum}_{i,t} \;=\; \frac{P_{i,t}}{P_{i,t-252}} - 1$$

  Higher values indicate stronger relative price trends.

- **Volatility (VOL)** – annualized standard deviation of daily returns over the past 12 months

  $$\text{Volatility}_{i,t} \;=\; \sigma\!\big(r_{i,t-252:t}\big)\,\sqrt{252}$$

  Lower volatility is preferred (low-risk stocks rank higher).

- **Profitability (ROE)** – rolling 12-month average of Return on Equity

  Higher values signal more profitable companies.

### Ranking methodology
1. For each trading date, calculate the raw factor values.
2. Assign stocks into **quintiles (1–5)** using `pd.qcut`.
   - Momentum, ROE: **descending** (5 = best).
   - Volatility: **ascending** (1 = highest risk, 5 = lowest risk).
3. Store the ranks in dedicated DataFrames (`mom_ranks`, `vol_ranks`, `roe_ranks`).

### Output
- **Rank matrices** for each factor (date × stock).
- **Raw factor time series** (12-month momentum, volatility, ROE).
- These scores will later be combined by the ML model to form stock weights.


## 4. 🧠 Model Training

In this section, we train a supervised machine-learning model to **predict optimal factor blend weights**.
The goal is to replace static allocations (e.g., 50/50 between two factors) with **time-varying weights** that adapt to market conditions.

### Methodology
1. **Target construction**
   - For each month in the training window, the *ex-post optimal factor weight* is determined via a **grid search**.
   - The grid search selects the weight combination that maximizes the **cumulative return** of the blended portfolio in the following month.
   - These optimal weights serve as the **training labels** for the ML model.

2. **Feature engineering**
   Features include:
   - Past factor returns (Momentum, Volatility, ROE)
   - Cross-sectional correlations between factors
   - Market-wide statistics (index return, index volatility)
   - Macro variables (risk-free rate, VIX)

   This ensures the model can capture both **factor-specific signals** and **market regime effects**.

3. **Learning algorithms**
   Different regressors are tested, including:
   - Random Forest Regressor
   - LightGBM
   - XGBoost
   - CatBoost

   The chosen algorithm is controlled by the parameter `ml_model` in the configuration.

4. **Model evaluation**
   After fitting the regressor on the training sample:
   - Goodness-of-fit is measured using $R^2$, MSE, and MAE.
   - **Feature importance** is extracted to understand which predictors are most influential.
   - **Univariate statistics** (F-scores and p-values) provide additional insight into predictor relevance.

### Output
- A trained ML model stored in the pipeline (`ml_model`).
- Performance metrics on the training set.
- Feature importance rankings.
- Statistical summary of predictor significance.

---

👉 The trained model will be applied in the next step to generate **monthly factor weights** in the out-of-sample period.


## 5. 🚀 Model Application

Once the machine-learning model has been trained, we apply it to the **out-of-sample period** to generate **monthly factor blend weights**.
These predicted weights are then used to construct stock-level portfolio allocations.

### Methodology
1. **Monthly prediction of factor weights**
   - For each month in the test period, factor and market features are recalculated.
   - The trained model predicts the optimal weight for the first factor (e.g., Momentum).
   - The second factor’s weight is determined as the complement (1 − predicted weight).
   - Predictions are clamped between 0 and 1 to ensure feasible allocations.

2. **Benchmarks for comparison**
   In addition to ML-driven weights, we construct two baseline strategies:
   - **50/50 allocation** – a static equal-weight blend of both factors.
   - **Average-ML allocation** – applies the *average ML-predicted weight* across the training set uniformly to all periods.

3. **Stock portfolio construction**
   - Factor ranks (Momentum, Volatility, ROE) are combined according to the predicted weights.
   - These integrated scores are multiplied with index weights and normalized to obtain final stock-level weights.
   - For benchmarks, the same procedure is followed with the static 50/50 and average-ML weights.
   - An additional **index portfolio** is included as a baseline.

### Output
- **Predicted factor weight series** for each month.
- **Benchmark weights** (50/50 and average-ML).
- **Stock-level portfolio weights** for ML, 50/50, average-ML, and index.
- A summary table of average factor weights across the test period.

---

👉 These stock weights form the input for the **backtest calculation** in the next step.


## 6. 📉 Backtest Calculation

In this step we simulate the **historical performance** of the ML-driven strategy and its benchmarks.
The backtest applies the monthly portfolio weights to realized stock returns and evaluates performance over the test period.

### Methodology
1. **Rebalancing frequency**
   - Portfolios are rebalanced at each month-end (or the chosen frequency in `params_`).
   - Within each rebalancing window, daily stock returns are aggregated based on the fixed weights determined at the start of the period.

2. **Strategies evaluated**
   - **ML Strategy** – portfolios constructed with ML-predicted factor weights.
   - **50/50 Strategy** – equal blend of both factors.
   - **Average-ML Strategy** – uses the average ML-predicted weight across the training sample.
   - **Index Strategy** – benchmark index weights without factor integration.

3. **Return calculation**
   - Daily portfolio returns are computed as the weighted sum of individual stock returns.
   - Cumulative performance is obtained by compounding daily returns.
   - Return series are stored for all strategies to enable direct comparison.

4. **Performance metrics**
   For each strategy we compute:
   - **Annualized return**
   - **Annualized volatility**
   - **Sharpe ratio**
   - **Maximum drawdown**

   These statistics summarize risk–return trade-offs and allow benchmarking between strategies.

### Output
- **Return time series** (daily and cumulative) for each strategy.
- **Performance summary table** with annualized metrics.
- Data prepared for visualization in the next section.

---

👉 The next step presents the **Results**, including visualizations, sensitivity checks, and a comparison across models and configurations.


## 7. 📊 Results

In this section we analyze and visualize the performance of the ML-driven strategy relative to the benchmarks.
The results are structured to provide insights into **time-period choices, feature selection, model specification, and factor relevance**.

### 7.1 📉 Choice of Training Period
- Investigates how the selection of in-sample training years affects out-of-sample performance.
- Compares performance across different training windows to check robustness.

### 7.2 📉 Choice of Feature Variables
- Evaluates which input features (e.g., factor returns, correlations, market volatility, macro variables) contribute most to predictive power.
- Uses feature importance and univariate statistics (F-scores, p-values) to assess relevance.

### 7.3 📉 Choice of Machine Learning Model
- Compares regressors (Random Forest, LightGBM, XGBoost, CatBoost).
- Benchmarks predictive accuracy and backtest performance.
- Highlights trade-offs between interpretability and predictive strength.

### 7.4 📉 Choice of Relevant Factors
- Tests different factor combinations (e.g., Momentum vs. ROE, Momentum vs. Volatility).
- Analyzes how factor pairings impact portfolio outcomes.

### 7.5 📉 Summary: Best Model
- Summarizes the best-performing configuration across all experiments.
- Reports final **risk–return metrics** (annualized return, volatility, Sharpe ratio, drawdown).
- Discusses interpretability (feature importance) and practical insights for factor investing.

---

📌 **Key takeaway:** Results show whether a **machine-learning-driven factor integration** can deliver **consistent outperformance** compared to static blends and the index benchmark, and under which conditions it performs best.


In [ ]:
results = run_ml_backtests([dict(name="Descriptives")], params_)

In [ ]:
df_dict = [res['GetStockScores'].return_stats for res in results]
display(df_dict[0]['basic_info'])
display(df_dict[0]['availability_stats'].style.format('{:.1f}'))

In [ ]:
for res in results:
    display(res['GetStockScores'].numb_companies_fig)

In [ ]:
df = pd.concat([res['GetStockScores'].factor_stats for res in results])
display(df.style.format('{:.1%}'))

In [ ]:
for res in results:
    display(res['GetStockScores'].cum_returns_fig)

In [ ]:
for res in results:
    display(res['GetStockScores'].rf_vix_fig)

In [ ]:
for res in results:
    figs_dict = res['GetStockScores'].vix_excess_return_figs
    for fig in figs_dict.values():
        display(fig)

In [ ]:
for res in results:
    figs_dict = res['GetStockScores'].rf_excess_returns_figs
    for fig in figs_dict.values():
        display(fig)

### 7.1 📉 Choice of Training Period

The choice of **training window** is critical for the stability and robustness of the ML model.
Since market regimes vary over time, different in-sample periods may lead to different model calibrations and therefore different out-of-sample performance.

#### Methodology
- Define alternative **training start and end dates** in the parameter dictionary (`params_`).
- Keep the **test window constant**, so that only the training horizon varies.
- Train separate models for each configuration and compare their backtest results.

#### Analysis
- Shorter training periods may better capture recent market dynamics but risk **overfitting** to short-lived patterns.
- Longer training periods provide more observations but may dilute signals from more recent market conditions.
- By comparing strategies across training windows, we can evaluate whether results are **robust** or highly sensitive to the sample choice.

#### Output
- **Performance tables** with annualized return, volatility, Sharpe ratio, and max drawdown across training periods.
- **Cumulative return plots** showing how the ML strategy evolves depending on the training window.
- Interpretation of whether certain windows provide systematically better predictive performance.

---

📌 **Key question:** Does extending or shortening the training sample materially change the ML strategy’s out-of-sample performance?


In [ ]:
# 1. Set model specifications
model_definitions = [
    dict(name="Model 1",
         training_end_period='2009-12-31',
         test_start_period='2010-01-01',
         ml_training_factors=['mom_roe_corr', 'past_mom_return', 'past_roe_return', 'past_index_return', 'past_index_vola', 'rf', 'vix']),
    dict(name="Model 2",
         training_end_period='2010-12-31',
         test_start_period='2011-01-01',
         ml_training_factors=['mom_roe_corr', 'past_mom_return', 'past_roe_return', 'past_index_return', 'past_index_vola', 'rf', 'vix']),
    dict(name="Model 3",
         training_end_period='2015-12-31',
         test_start_period='2016-01-01',
         ml_training_factors=['mom_roe_corr', 'past_mom_return', 'past_roe_return', 'past_index_return', 'past_index_vola', 'rf', 'vix']),
    dict(name="Model 4",
         training_end_period='2020-12-31',
         test_start_period='2021-01-01',
         ml_training_factors=['mom_roe_corr', 'past_mom_return', 'past_roe_return', 'past_index_return', 'past_index_vola', 'rf', 'vix'])
]

# 2. Run models
results = run_ml_backtests(model_definitions, params_)

# 3. Display results
print("Average Factor Weights over Time)")
df = pd.concat([res['AverageFactorWeights'] for res in results])
display(df.style.format('{:.1%}'))
print("Goodness of model (in-sample-test)")
df = pd.concat([res['GoodnessOfModel'] for res in results])
display(df.style.format('{:.1%}'))
print("Goodness of model (out-of-sample-test)")
df = pd.concat([res['ExcessReturns'] for res in results])
display(df.style.format('{:.2%}'))
print("Feature Importance")
df = pd.concat([res['FeatureImportance'] for res in results])
display(df.style.format('{:.2f}'))
print("Feature Stats")
df = pd.concat([res['FeatureStats'] for res in results])
display(df.style.format('{:.2f}'))

### 7.2 📉 Choice of Feature Variables

The predictive power of the ML model depends heavily on the **input features** used during training.
This section evaluates the role of different variable groups in explaining and forecasting optimal factor weights.

#### Methodology
- Use the parameter `ml_training_factors` to specify alternative feature sets.
- Train separate models with different feature combinations (e.g., only factor returns vs. full set with macro variables).
- Compare predictive accuracy and backtest performance across setups.

#### Types of features
- **Factor-specific signals**
  - Past returns of Momentum, Volatility, ROE portfolios
  - Cross-sectional correlations between factors

- **Market-level statistics**
  - Past index return and volatility
  - Dispersion of factor scores

- **Macro/regime indicators**
  - Risk-free rate (RF)
  - Volatility index (VIX)

#### Analysis
- Feature importance plots highlight which variables have the strongest explanatory power.
- Univariate statistics (F-scores, p-values) indicate which features are statistically significant.
- Removing or adding features shows whether predictive performance is robust or overly dependent on specific signals.

#### Output
- **Feature importance rankings** across models.
- **Performance comparison tables** for different feature sets.
- Interpretation of whether macro features (RF, VIX) improve predictive accuracy beyond pure factor information.

---

📌 **Key question:** Which features truly drive the predictive power of the ML model, and are the results stable across different feature specifications?


In [ ]:
# 1. Set model specifications
model_definitions = [
    dict(name="Model 5",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['mom_roe_corr', 'past_mom_return', 'past_roe_return']),
    dict(name="Model 6",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['past_mom_return', 'past_roe_return']),
    dict(name="Model 7",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['rf', 'vix']),
    dict(name="Model 8",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['past_index_return', 'past_index_vola']),
    dict(name="Model 9",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['rf']),
    dict(name="Model 10",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['vix'])
    ]

# 2. Run models
results = run_ml_backtests(model_definitions, params_)

# 3. Display results
print("Average Factor Weights over Time)")
df = pd.concat([res['AverageFactorWeights'] for res in results])
display(df.style.format('{:.1%}'))
print("Goodness of model (in-sample-test)")
df = pd.concat([res['GoodnessOfModel'] for res in results])
display(df.style.format('{:.1%}'))
print("Goodness of model (out-of-sample-test)")
df = pd.concat([res['ExcessReturns'] for res in results])
display(df.style.format('{:.2%}'))
print("Feature Importance")
df = pd.concat([res['FeatureImportance'] for res in results])
display(df.style.format('{:.2f}'))
print("Feature Stats")
df = pd.concat([res['FeatureStats'] for res in results])
display(df.style.format('{:.2f}'))

### 7.3 📉 Choice of Machine Learning Model

The choice of learning algorithm can significantly influence predictive accuracy, interpretability, and computational efficiency.
In this section we compare different regression models for predicting factor blend weights.

#### Methodology
- Specify the algorithm through the parameter `ml_model`.
- Train models using identical training data and feature sets, changing only the algorithm.
- Evaluate both in-sample fit (goodness-of-model metrics) and out-of-sample backtest performance.

#### Models considered
- **Random Forest Regressor**
  - Non-parametric, handles non-linearities well, robust to noise.
- **LightGBM**
  - Gradient boosting on decision trees, efficient and scalable, good for tabular data.
- **XGBoost**
  - Widely used gradient boosting framework, strong predictive performance, but heavier computational cost.
- **CatBoost**
  - Gradient boosting with categorical feature handling, avoids overfitting through ordered boosting.

#### Analysis
- Compare metrics such as $R^2$, MSE, and MAE on the training sample.
- Evaluate out-of-sample performance (annualized return, volatility, Sharpe ratio, drawdown).
- Contrast interpretability: Random Forest provides intuitive feature importances, while boosting models may capture more complex patterns but are less transparent.

#### Output
- **Performance summary tables** across ML models.
- **Feature importance plots** for each algorithm.
- Insights on the trade-off between interpretability and predictive accuracy.

---

📌 **Key question:** Which algorithm provides the best balance of predictive performance, robustness, and interpretability for factor weight prediction?


In [ ]:
# 1. Set model specifications
model_definitions = [
    dict(name="Model 11",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['rf'],
         ml_model='RandomForestRegressor'),
    dict(name="Model 12",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['rf'],
         ml_model='lightgbm'),
    dict(name="Model 13",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['rf'],
         ml_model='xgboost'),
    dict(name="Model 14",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['rf'],
         ml_model='catboost')
]

# 2. Run models
results = run_ml_backtests(model_definitions, params_)

# 3. Display results
print("Average Factor Weights over Time)")
df = pd.concat([res['AverageFactorWeights'] for res in results])
display(df.style.format('{:.1%}'))
print("Goodness of model (in-sample-test)")
df = pd.concat([res['GoodnessOfModel'] for res in results])
display(df.style.format('{:.1%}'))
print("Goodness of model (out-of-sample-test)")
df = pd.concat([res['ExcessReturns'] for res in results])
display(df.style.format('{:.2%}'))
print("Feature Importance")
df = pd.concat([res['FeatureImportance'] for res in results])
display(df.style.format('{:.2f}'))
print("Feature Stats")
df = pd.concat([res['FeatureStats'] for res in results])
display(df.style.format('{:.2f}'))

### 7.4 📉 Choice of Relevant Factors

The predictive task also depends on **which factors** are chosen for integration.
This section explores how different factor combinations affect portfolio construction and performance.

#### Methodology
- Specify factor pairs through the parameter `relevant_factors` (e.g., `['mom', 'roe']`, `['mom', 'vol']`).
- Train and apply the ML model separately for each factor combination.
- Compare results across different setups to analyze sensitivity to factor choice.

#### Factor combinations tested
- **Momentum (MOM) vs. Profitability (ROE)**
  - Captures the trade-off between trend persistence and fundamental quality.
- **Momentum (MOM) vs. Volatility (VOL)**
  - Balances return-seeking behavior with risk control.
- **Volatility (VOL) vs. Profitability (ROE)**
  - Explores whether low-risk and high-quality factors provide complementary signals.

#### Analysis
- Compare the average ML-predicted weights across test periods for each factor pair.
- Evaluate whether certain factor pairings systematically outperform others.
- Assess stability: do factor weights shift strongly over time, or remain relatively stable?

#### Output
- **Time series of factor weights** for each combination.
- **Performance comparison tables** across factor pairs.
- Interpretation of which factors are best suited for ML-driven integration.

---

📌 **Key question:** Which factor pairs benefit most from machine learning–based dynamic weighting, and under which market conditions?


In [ ]:
# 1. Set model specifications
model_definitions = [
    dict(name="Model 15",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['rf'],
         relevant_factors = ['mom', 'roe'],
         ml_model='xgboost'),
    dict(name="Model 16",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['rf'],
         relevant_factors = ['mom', 'vol'],
         ml_model='xgboost'),
    dict(name="Model 17",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['rf'],
         relevant_factors = ['roe', 'vol'],
         ml_model='xgboost')
]

# 2. Run models
results = run_ml_backtests(model_definitions, params_)

# 3. Display results
print("Average Factor Weights over Time)")
df = pd.concat([res['AverageFactorWeights'] for res in results])
display(df.style.format('{:.1%}'))
print("Goodness of model (in-sample-test)")
df = pd.concat([res['GoodnessOfModel'] for res in results])
display(df.style.format('{:.1%}'))
print("Goodness of model (out-of-sample-test)")
df = pd.concat([res['ExcessReturns'] for res in results])
display(df.style.format('{:.2%}'))
print("Feature Importance")
df = pd.concat([res['FeatureImportance'] for res in results])
display(df.style.format('{:.2f}'))
print("Feature Stats")
df = pd.concat([res['FeatureStats'] for res in results])
display(df.style.format('{:.2f}'))

### 7.5 📉 Summary: Best Model

After testing different **training periods, feature sets, ML algorithms, and factor combinations**, we summarize the results to identify the best-performing configuration.

#### Methodology
- Aggregate results from all experiments into comparison tables.
- Focus on both **statistical fit** (R², MSE, MAE) and **economic performance** (annualized return, volatility, Sharpe ratio, max drawdown).
- Evaluate stability of results across time and robustness to parameter choices.

#### Key criteria
- **Performance**: Did the ML-driven strategy outperform both the 50/50 baseline and the index on a risk-adjusted basis?
- **Robustness**: Are the results consistent across different training samples and feature specifications?
- **Interpretability**: Do the feature importance rankings align with economic intuition (e.g., VIX relevant in crisis periods, ROE spreads relevant in expansions)?

#### Output
- A consolidated **performance table** ranking all tested models.
- **Visualization of cumulative returns** for the top-performing ML strategy vs. benchmarks.
- Discussion of why the best model outperformed and under which conditions.

---

📌 **Final takeaway:** The best model is not only the one with the highest backtest return, but the one that achieves **robust, interpretable, and consistent outperformance** relative to static allocations and the index benchmark.


In [ ]:
# 1. Set model specifications
model_definitions = [
    dict(name="Model 18",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['rf'],
         relevant_factors = ['roe', 'vol'],
         ml_model='xgboost')
]

# 2. Run models
results = run_ml_backtests(model_definitions, params_)

# 3. Display results
print("Average Factor Weights over Time)")
df = pd.concat([res['AverageFactorWeights'] for res in results])
display(df.style.format('{:.1%}'))
print("Goodness of model (in-sample-test)")
df = pd.concat([res['GoodnessOfModel'] for res in results])
display(df.style.format('{:.1%}'))
print("Goodness of model (out-of-sample-test)")
df = pd.concat([res['ExcessReturns'] for res in results])
display(df.style.format('{:.2%}'))
print("Feature Importance")
df = pd.concat([res['FeatureImportance'] for res in results])
display(df.style.format('{:.2f}'))
print("Feature Stats")
df = pd.concat([res['FeatureStats'] for res in results])
display(df.style.format('{:.2f}'))

## 8. ✅ Conclusion & Next Steps

### Conclusion
This project demonstrated how **machine learning can be applied to dynamic factor integration** within a systematic equity strategy.
Key insights include:
- ML-based models are able to generate **time-varying factor weights** that adapt to changing market regimes.
- Out-of-sample results suggest that the ML-driven strategy can outperform static benchmarks (50/50 and average-ML) as well as the index in terms of risk-adjusted performance.
- Feature importance analysis highlights that both **factor spreads** and **macro indicators** (e.g., VIX, risk-free rate) play a crucial role in predicting optimal factor weights.
- Robustness checks across different training periods and factor combinations confirm that results are not solely sample-specific.

### Limitations
- Performance depends on **data quality and availability** (e.g., survivorship bias, missing ROE data).
- The model is limited to **two-factor blends**; expanding to multiple factors may increase complexity and overfitting risk.
- Backtests are historical simulations and may not fully capture transaction costs or liquidity constraints.

### Next Steps
- Extend the framework to include **more factors** (e.g., Value, Size, Quality).
- Test alternative ML models (e.g., neural networks, ensemble approaches).
- Incorporate **transaction costs and turnover constraints** for more realistic performance assessment.
- Explore **explainability tools** (e.g., SHAP values) to improve interpretability of ML predictions.
- Deploy the pipeline in a **live environment** to monitor real-time predictions and evaluate practical feasibility.

---

📌 **Final note:** Machine learning offers a promising path for factor investing, but careful validation, robustness checks, and economic intuition remain essential to ensure reliable and actionable results.
